In [ ]:
# =====================================================================
# الخطوة 1 المحدثة: تحميل قاعدة بيانات MIT-BIH الكاملة وإعداد الـ DataLoaders
# =====================================================================
from google.colab import drive
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

print("جاري ربط حساب Google Drive للوصول إلى البيانات المعتمدة...")
drive.mount('/content/drive')

csv_path = '/content/drive/MyDrive/MedNetLite_Project/mitbih_beats_binary.csv'

if os.path.exists(csv_path):
    print("✓ تم العثور على قاعدة البيانات المعتمدة بنجاح.")

    # 1. قراءة قاعدة البيانات بالكامل (حذف nrows=5 لقراءة كافة المرضى)
    print("⏳ جاري قراءة كامل بيانات MIT-BIH الطبية إلى الذاكرة...")
    df = pd.read_csv(csv_path)
    print(f"✅ تم تحميل البيانات بنجاح! الحجم الإجمالي الفعلي: {df.shape}")

    # 2. فصل الميزات (الإشارات) عن عمود التصنيف (الهدف)
    # نفترض أن العمود الأخير هو الـ label (التصنيف الثنائي 0 أو 1)
    X_raw = df.iloc[:, :-1].values.astype(np.float32)
    y_raw = df.iloc[:, -1].values.astype(np.int64)

    # تحسين الأبعاد لتتوافق مع طبقة الـ Conv1D (حجم الدفعة، القنوات=1، طول الإشارة=300)
    if X_raw.shape[1] != 300:
        print(f"💡 تنبيه: طول الإشارة في الملف هو {X_raw.shape[1]}، سيتم ضبط المدخل الوهمي لاحقاً ليتوافق معه.")
    X_raw = np.expand_dims(X_raw, axis=1)

    # 3. تقسيم البيانات إلى 80% تدريب و 20% اختبار مع الحفاظ على نسب الفئات (Stratify)
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
    )

    # 4. تحويل المصفوفات إلى Tensors الخاصة بـ PyTorch
    X_train_tensor = torch.tensor(X_train)
    y_train_tensor = torch.tensor(y_train)
    X_test_tensor = torch.tensor(X_test)
    y_test_tensor = torch.tensor(y_test)

    # 5. إنشاء الـ TensorDataset والـ DataLoaders الحقيقية للتدريب
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"🔹 إجمالي عينات التدريب الحقيقية: {len(train_dataset)}")
    print(f"🔹 إجمالي عينات الاختبار الحقيقية: {len(test_dataset)}")
    print(f"✅ الـ DataLoaders جاهزة تماماً الآن للتدريب الفعلي!")
else:
    print("⚠️ لم يتم العثور على الملف في المسار المحدّد، يرجى التأكد من وجوده في الدرايف.")

جاري ربط حساب Google Drive للوصول إلى البيانات المعتمدة...
Mounted at /content/drive
✓ تم العثور على قاعدة البيانات المعتمدة بنجاح.
⏳ جاري قراءة كامل بيانات MIT-BIH الطبية إلى الذاكرة...
✅ تم تحميل البيانات بنجاح! الحجم الإجمالي الفعلي: (101406, 301)
🔹 إجمالي عينات التدريب الحقيقية: 81124
🔹 إجمالي عينات الاختبار الحقيقية: 20282
✅ الـ DataLoaders جاهزة تماماً الآن للتدريب الفعلي!


In [ ]:
# =====================================================================
# الخطوة 2: تثبيت مكتبات النموذج والتقسيم الإحصائي الصارم المانع للتسريب
# =====================================================================
print("جاري تثبيت وإعداد مكتبات التعلم والتدريب...")
!pip install -q scikit-learn torch torchvision

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# 1. تحميل قاعدة البيانات بالكامل إلى الذاكرة
print("جاري تحميل قاعدة البيانات الكاملة لتقسيمها إحصائياً...")
full_df = pd.read_csv(csv_path)

X = full_df.iloc[:, :-1].values.astype(np.float32)  # المصفوفات الممثلة للنبضات (300 عينة)
y = full_df.iloc[:, -1].values.astype(np.int64)     # التسميات الثنائية (0 أو 1)

# 2. التقسيم الإحصائي المعتمد (80% تدريب وتطوير، 20% اختبار نهائي)
# نستخدم stratify=y لضمان الحفاظ على نفس نسبة الفئات المرضية في المجموعات
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. تقسيم مجموعة التدريب والتطوير إلى (80% تدريب فعلي، 20% تحقق وتأكيد أثناء الدورات)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, random_state=42, stratify=y_train_val
)

print("\n" + "="*50)
print("📊 تقرير أبعاد ومجموعات البيانات بعد التقسيم الصارم:")
print("="*50)
print(f"🔹 مجموعة التدريب الفعلي (Train): {X_train.shape[0]:,} نبضة")
print(f"   [طبيعي: {(y_train==0).sum():,} | مرضي: {(y_train==1).sum():,}]")
print(f"🔹 مجموعة التحقق والتأكيد (Validation): {X_val.shape[0]:,} نبضة")
print(f"   [طبيعي: {(y_val==0).sum():,} | مرضي: {(y_val==1).sum():,}]")
print(f"🔹 مجموعة الاختبار النهائي (Test): {X_test.shape[0]:,} نبضة")
print(f"   [طبيعي: {(y_test==0).sum():,} | مرضي: {(y_test==1).sum():,}]")
print("="*50)
print("✓ تم التقسيم وتثبيت الحالة العشوائية (Seed=42) لضمان إعادة الإنتاجية الكاملة.")

جاري تثبيت وإعداد مكتبات التعلم والتدريب...
جاري تحميل قاعدة البيانات الكاملة لتقسيمها إحصائياً...

📊 تقرير أبعاد ومجموعات البيانات بعد التقسيم الصارم:
🔹 مجموعة التدريب الفعلي (Train): 64,899 نبضة
   [طبيعي: 57,976 | مرضي: 6,923]
🔹 مجموعة التحقق والتأكيد (Validation): 16,225 نبضة
   [طبيعي: 14,494 | مرضي: 1,731]
🔹 مجموعة الاختبار النهائي (Test): 20,282 نبضة
   [طبيعي: 18,118 | مرضي: 2,164]
✓ تم التقسيم وتثبيت الحالة العشوائية (Seed=42) لضمان إعادة الإنتاجية الكاملة.


In [ ]:
# =====================================================================
# الخلية 1 المحدثة: معمارية MedNetLite المتوافقة بالكامل مع تصدير ONNX
# =====================================================================
import torch
import torch.nn as nn

class CBAM_1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM_1D, self).__init__()
        # 1. Channel Attention
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid_channel = nn.Sigmoid()

        # 2. Spatial Attention
        self.conv_spatial = nn.Conv1d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        # --- Channel Attention ---
        b, c, n = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c)).view(b, c, 1)

        # الحل اليدوي المتوافق مع ONNX بدلاً من الـ AdaptiveMaxPool1d القديم
        max_out_val, _ = torch.max(x, dim=2, keepdim=True)
        max_out = self.fc(max_out_val.view(b, c)).view(b, c, 1)

        channel_out = self.sigmoid_channel(avg_out + max_out)
        x = x * channel_out

        # --- Spatial Attention ---
        avg_spatial = torch.mean(x, dim=1, keepdim=True)

        # الحل اليدوي لـ Spatial Max Pool المتوافق مع ONNX
        max_spatial, _ = torch.max(x, dim=1, keepdim=True)

        spatial_in = torch.cat([avg_spatial, max_spatial], dim=1)
        spatial_out = self.sigmoid_spatial(self.conv_spatial(spatial_in))
        x = x * spatial_out

        return x

class MedNetLite(nn.Module):
    def __init__(self, num_classes=2):
        super(MedNetLite, self).__init__()
        # الشبكة الأساسية لاستخلاص الميزات من الإشارات الطبية
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            CBAM_1D(16),

            nn.Conv1d(16, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            CBAM_1D(32),

            nn.AdaptiveAvgPool1d(1)
        )
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# تجهيز كائن النموذج واختبار البيئة (على الـ GPU إن وجد)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedNetLite().to(device)
print(f"✅ تم بناء النموذج بنجاح وتجهيزه على الجهاز: {device}")

✅ تم بناء النموذج بنجاح وتجهيزه على الجهاز: cpu


In [ ]:
# =====================================================================
# الخطوة الأخيرة: حفظ الأوزان المدربة للنموذج القياسي في Google Drive
# =====================================================================
import os

# تحديد مسار الحفظ داخل المجلد المشترك في Drive
drive_models_dir = '/content/drive/MyDrive/MedNetLite_Project/Models'
os.makedirs(drive_models_dir, exist_ok=True)

model_save_path = os.path.join(drive_models_dir, 'mednetlite_standard_weights.pth')

# حفظ قاموس الأوزان (State Dict) الخاص بالنموذج
torch.save(model.state_dict(), model_save_path)

if os.path.exists(model_save_path):
    print("="*50)
    print("📢 تم حفظ أوزان نموذج MedNetLite المدرب بنجاح وأمان!")
    print(f"📁 مسار ملف الأوزان المستهدف: {model_save_path}")
    print("="*50)
else:
    print("⚠️ تحذير: فشل حفظ الملف، يرجى التحقق من الاتصال بالـ Drive.")

📢 تم حفظ أوزان نموذج MedNetLite المدرب بنجاح وأمان!
📁 مسار ملف الأوزان المستهدف: /content/drive/MyDrive/MedNetLite_Project/Models/mednetlite_standard_weights.pth


In [7]:
# =====================================================================
# الكود الختامي الموحد والمحدث بالكامل للمفكرة الثانية (Notebook 2)
# =====================================================================
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from google.colab import drive

# 1. ربط Google Drive وتحميل البيانات
print("⏳ جاري ربط حساب Google Drive وتحميل البيانات الطبية...")
drive.mount('/content/drive')

csv_path = '/content/drive/MyDrive/MedNetLite_Project/mitbih_beats_binary.csv'

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ لم يتم العثور على قاعدة البيانات في المسار المعتمد: {csv_path}")

df = pd.read_csv(csv_path)
print(f"✅ تم تحميل قاعدة البيانات بنجاح! الحجم الإجمالي: {df.shape}")

# فصل الميزات والتصنيف وضبط الأبعاد لـ Conv1D (Batch, Channels=1, Length)
X_raw = df.iloc[:, :-1].values.astype(np.float32)
y_raw = df.iloc[:, -1].values.astype(np.int64)

# تحديد عدد الفئات ديناميكياً
num_classes = len(np.unique(y_raw))
print(f"📊 عدد الفئات المستهدفة: {num_classes}")

X_raw = np.expand_dims(X_raw, axis=1)

# تقسيم البيانات (80% تدريب، 20% اختبار)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# حساب أوزان الفئات المنعمة (Smoothed Class Weights) لضمان استقرار التدريب
classes = np.unique(y_train)
raw_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
smoothed_weights = np.sqrt(raw_weights)
class_weights_tensor = torch.tensor(smoothed_weights, dtype=torch.float32)

# إنشاء الـ DataLoaders
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=256, shuffle=False)
print(f"🔹 عينات التدريب: {X_train.shape[0]} | عينات الاختبار: {X_test.shape[0]}")


# 2. تعريف معمارية MedNetLite المحسنة والآمنة عدديًا
class CBAM1D(nn.Module):
    def __init__(self, channels, reduction=2): # تعديل reduction لمنع zero-element tensors
        super(CBAM1D, self).__init__()
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels, bias=False)
        )
        self.conv_spatial = nn.Conv1d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, l = x.size()
        mean_pool = x.view(b, c, -1).mean(dim=2)
        max_pool = torch.max(x.view(b, c, -1), dim=2)[0]

        ca_weight = self.sigmoid(self.fc(mean_pool) + self.fc(max_pool)).view(b, c, 1)
        x = x * ca_weight

        sa_mean = x.mean(dim=1, keepdim=True)
        sa_max = torch.max(x, dim=1, keepdim=True)[0]
        sa_weight = self.sigmoid(self.conv_spatial(torch.cat([sa_mean, sa_max], dim=1)))

        return x * sa_weight, sa_weight


class MedNetLite(nn.Module):
    def __init__(self, num_classes=5):
        super(MedNetLite, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=15, stride=2, padding=7, bias=True)
        self.bn1 = nn.BatchNorm1d(16)
        self.cbam1 = CBAM1D(16)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=7, stride=2, padding=3, bias=True)
        self.bn2 = nn.BatchNorm1d(32)
        self.cbam2 = CBAM1D(32)

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(32, num_classes)
        self.last_sa_weights = None

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x, _ = self.cbam1(x)

        x = torch.relu(self.bn2(self.conv2(x)))
        x, sa_weights = self.cbam2(x)
        self.last_sa_weights = sa_weights

        x = self.pool(x).view(x.size(0), -1)
        return self.classifier(x)


# 3. دورة التدريب المعايرة
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedNetLite(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

print(f"\n🎯 بدء التدريب المعاير على الـ {device}...")
print("-" * 60)

epochs = 15
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    start_time = time.time()

    for signals, labels in train_loader:
        signals, labels = signals.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(signals)
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * signals.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()
    acc = (correct / total) * 100
    print(f"Epoch [{epoch+1:02d}/{epochs}] - Loss: {running_loss/total:.4f} - Train Acc: {acc:.2f}% - Time: {time.time()-start_time:.2f}s")

print("-" * 60)
print("🎉 انتهى تدريب النموذج الرئيسي بنجاح وبطريقة مطابقة للبروتوكول!")

# 4. حفظ الأوزان الذهبية
save_path = '/content/drive/MyDrive/MedNetLite_Project/mednetlite_standard_weights.pth'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)
print(f"💾 تم حفظ النموذج الرئيسي بنجاح في:\n ➡️ {save_path}")

⏳ جاري ربط حساب Google Drive وتحميل البيانات الطبية...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ تم تحميل قاعدة البيانات بنجاح! الحجم الإجمالي: (101406, 301)
📊 عدد الفئات المستهدفة: 2
🔹 عينات التدريب: 81124 | عينات الاختبار: 20282

🎯 بدء التدريب المعاير على الـ cpu...
------------------------------------------------------------
Epoch [01/15] - Loss: 0.2832 - Train Acc: 94.05% - Time: 23.93s
Epoch [02/15] - Loss: 0.1845 - Train Acc: 96.44% - Time: 25.29s
Epoch [03/15] - Loss: 0.1537 - Train Acc: 96.90% - Time: 24.32s
Epoch [04/15] - Loss: 0.1411 - Train Acc: 97.14% - Time: 24.59s
Epoch [05/15] - Loss: 0.1302 - Train Acc: 97.39% - Time: 25.01s
Epoch [06/15] - Loss: 0.1202 - Train Acc: 97.58% - Time: 23.50s
Epoch [07/15] - Loss: 0.1133 - Train Acc: 97.77% - Time: 25.08s
Epoch [08/15] - Loss: 0.1077 - Train Acc: 97.89% - Time: 24.60s
Epoch [09/15] - Loss: 0.1027 - Train Acc: 97.90% - Time: 23.92s
Epoch

In [8]:
# =====================================================================
# كود التحقق المتقاطع المعاير والمطابق تماماً (5-Fold Cross-Validation)
# =====================================================================
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

# 1. التأكد من تجهيز البيانات
if 'X_raw' in globals() and 'y_raw' in globals():
    X_data, y_data = X_raw, y_raw
elif 'X_train_full' in globals():
    X_data, y_data = X_train_full, y_train_full
elif 'X_train' in globals() and 'y_train' in globals():
    X_data, y_data = X_train, y_train

if isinstance(X_data, torch.Tensor):
    X_data = X_data.cpu().numpy()
if isinstance(y_data, torch.Tensor):
    y_data = y_data.cpu().numpy()

num_classes = len(np.unique(y_data))

# 2. حساب أوزان الفئات المنعمة (Smoothed Weights)
classes = np.unique(y_data)
raw_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_data)
smoothed_weights = np.sqrt(raw_weights)
class_weights_tensor = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies, fold_precisions, fold_recalls, fold_f1s = [], [], [], []

print(f"🚀 بدء بروتوكول 5-Fold المعاير والمطابق للنموذج الرئيسي (15 Epochs لكل Fold)...")
print(f"📊 عدد الفئات: {num_classes} | إجمالي عينات البيانات: {len(y_data)}")
print("=" * 65)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data)):
    print(f"\n🔄 --- [Fold {fold + 1}/5] ---")

    X_tr, y_tr = X_data[train_idx], y_data[train_idx]
    X_va, y_va = X_data[val_idx], y_data[val_idx]

    # التأكد من الأبعاد المطلوبة لـ Conv1D (Batch, Channels, Length)
    if len(X_tr.shape) == 2:
        X_tr_t = torch.tensor(X_tr, dtype=torch.float32).unsqueeze(1)
        X_va_t = torch.tensor(X_va, dtype=torch.float32).unsqueeze(1)
    else:
        X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
        X_va_t = torch.tensor(X_va, dtype=torch.float32)

    y_tr_t = torch.tensor(y_tr, dtype=torch.long)
    y_va_t = torch.tensor(y_va, dtype=torch.long)

    train_dataset = torch.utils.data.TensorDataset(X_tr_t, y_tr_t)
    val_dataset = torch.utils.data.TensorDataset(X_va_t, y_va_t)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=256, shuffle=False)

    # إعادة تهيئة نموذج MedNetLite بالهيكلية الجديدة
    model = MedNetLite(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

    best_val_f1 = 0.0
    best_model_path = f"best_mednet_fold_{fold+1}.pth"

    for epoch in range(15):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(x_batch)
            loss = criterion(out, y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()

        # تقييم الـ Validation بناءً على Macro F1
        model.eval()
        epoch_preds, epoch_targets = [], []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(device)
                out = model(x_batch)
                preds = torch.argmax(out, dim=1).cpu().numpy()
                epoch_preds.extend(preds)
                epoch_targets.extend(y_batch.numpy())

        _, _, ep_f1, _ = precision_recall_fscore_support(epoch_targets, epoch_preds, average='macro', zero_division=0)

        if ep_f1 > best_val_f1:
            best_val_f1 = ep_f1
            torch.save(model.state_dict(), best_model_path)

    # تحميل وتقييم أفضل نموذج تم حفظه للطية
    model.load_state_dict(torch.load(best_model_path))
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            out = model(x_batch)
            preds = torch.argmax(out, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(y_batch.numpy())

    acc = accuracy_score(all_targets, all_preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)

    fold_accuracies.append(acc)
    fold_precisions.append(prec * 100)
    fold_recalls.append(rec * 100)
    fold_f1s.append(f1 * 100)

    print(f"✅ Fold {fold + 1} - Best Acc: {acc:.2f}% | Macro F1: {f1*100:.2f}%")

# ============================================================
# 📊 التقرير النهائي للتحقق المتقاطع
# ============================================================
print("\n" + "=" * 65)
print("📊 التقرير النهائي المعاير للتحقق المتقاطع (5-Fold CV Report)")
print("=" * 65)
print(f"🔹 Accuracy:  {np.mean(fold_accuracies):.2f}% ± {np.std(fold_accuracies):.2f}%")
print(f"🔹 Precision: {np.mean(fold_precisions):.2f}% ± {np.std(fold_precisions):.2f}%")
print(f"🔹 Recall:    {np.mean(fold_recalls):.2f}% ± {np.std(fold_recalls):.2f}%")
print(f"🔹 F1-Score:  {np.mean(fold_f1s):.2f}% ± {np.std(fold_f1s):.2f}%")
print("=" * 65)

🚀 بدء بروتوكول 5-Fold المعاير والمطابق للنموذج الرئيسي (15 Epochs لكل Fold)...
📊 عدد الفئات: 2 | إجمالي عينات البيانات: 101406

🔄 --- [Fold 1/5] ---
✅ Fold 1 - Best Acc: 98.26% | Macro F1: 95.29%

🔄 --- [Fold 2/5] ---
✅ Fold 2 - Best Acc: 98.33% | Macro F1: 95.57%

🔄 --- [Fold 3/5] ---
✅ Fold 3 - Best Acc: 98.25% | Macro F1: 95.35%

🔄 --- [Fold 4/5] ---
✅ Fold 4 - Best Acc: 98.18% | Macro F1: 95.12%

🔄 --- [Fold 5/5] ---
✅ Fold 5 - Best Acc: 98.17% | Macro F1: 95.17%

📊 التقرير النهائي المعاير للتحقق المتقاطع (5-Fold CV Report)
🔹 Accuracy:  98.24% ± 0.06%
🔹 Precision: 96.14% ± 0.48%
🔹 Recall:    94.50% ± 0.46%
🔹 F1-Score:  95.30% ± 0.16%
